# 02 — NDVI Analysis & Crop Health Classification

Classifies each constituency by crop activity level based on mean NDVI.

**Classification thresholds:**
- ≥ 0.28 → High Activity (active irrigation / perennial crops)
- 0.25–0.28 → Moderate Activity
- < 0.25 → Low Activity / Post-Harvest (fallow or stressed)

> **Seasonal context:** March 2026 is post-Samba harvest season.
> NDVI values reflect the transition period — green pixels are irrigated
> perennial crops (sugarcane, gardens) not yet-to-harvest rice.

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

with open('../data/processed/ndvi_per_constituency.json') as f:
    data = json.load(f)
df = pd.DataFrame(data['constituencies'])
df = df.sort_values('mean_ndvi', ascending=False).reset_index(drop=True)

print('Constituency Crop Health Rankings:')
print(f"{'Rank':<5}{'Constituency':<22}{'Mean NDVI':<12}{'Class'}")
print('-'*52)
for i, row in df.iterrows():
    print(f"{i+1:<5}{row.constituency:<22}{row.mean_ndvi:<12.4f}{row.crop_health}")

print(f"\nDistrict mean NDVI: {df.mean_ndvi.mean():.4f}")
print(f"Highest: {df.iloc[0].constituency} ({df.iloc[0].mean_ndvi:.4f})")
print(f"Lowest:  {df.iloc[-1].constituency} ({df.iloc[-1].mean_ndvi:.4f})")

## Ranking Chart

In [ ]:
colors_map = {
    'High Activity':'#1a9850',
    'Moderate Activity':'#91cf60',
    'Low Activity / Post-Harvest':'#fc8d59'
}
bar_colors = [colors_map[h] for h in df.crop_health]

fig, ax = plt.subplots(figsize=(9,5))
bars = ax.barh(df.constituency[::-1], df.mean_ndvi[::-1],
               color=bar_colors[::-1], edgecolor='white', height=0.65)
for bar, val in zip(bars, df.mean_ndvi[::-1]):
    ax.text(val+0.002, bar.get_y()+bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=9)
ax.set_xlabel('Mean NDVI (March 2026)', fontsize=10)
ax.set_title('Thanjavur — Crop Activity by Constituency\nSentinel-2 NDVI, 25 March 2026',
             fontsize=11, fontweight='bold')
patches = [mpatches.Patch(color=colors_map[k], label=k) for k in colors_map]
ax.legend(handles=patches, fontsize=8, loc='lower right')
for sp in ['top','right']: ax.spines[sp].set_visible(False)
plt.tight_layout()
plt.savefig('../assets/thanjavur_ndvi_ranking.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved')